In [63]:
#This makes the summary stats using 050_Savings_To_Dollars
import pandas as pd
import numpy as np

In [64]:
# this code reads in the massive cost dataframe 


### References
# this code takes the raw links as strings and does webscrpaing like easy bid to get references for everything in APA format

In [ ]:
# Install required packages if needed
#!pip install beautifulsoup4 requests lxml python-docx

In [ ]:
references_df = pd.read_excel("050_input/demo_reference_list.xlsx")

In [ ]:
# this code creates a bibliography of references used in the model
# there is a row in each workpaper called reference that contains the citation for the data source
# the value is a link to where the information referenced is from
# we need to take that that link and put it into apa format

#libraries
# Parse existing references to create a URL lookup dictionary
import re
from urllib.parse import urlparse, urlunparse
from bs4 import BeautifulSoup
import requests
from datetime import datetime
import json
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH


In [ ]:
# Pre-formatted APA references with URLs
existing_references = """
Census Reporter. (n.d.). Census Tabulation Detail: Housing Units. Retrieved September 2025, from https://censusreporter.org/: https://censusreporter.org/tables/B25001/

Commission, M. P. (2024). 2024 Energy Waste Reduction Report to the Legislature. Michigan Department of Licensing and Regulatory Affairs. Retrieved September 2025, from https://www.michigan.gov/mpsc/-/media/Project/Websites/mpsc/regulatory/reports/pa295-ewr/2024-EWR-Report-to-theLegislature.pdf 

Ellsworth, A. D. (2021). Michigan Baseline Housing Study [Final report]. Cadmus. Retrieved from https://www.michigan.gov/mpsc/-/media/Project/Websites/mpsc/regulatory/ewr/MEMDand-BRM/Michigan-Baseline-Housing-StudyFinal-2021-05-25.pdf

Gagnon, P. (2024). Workbooks for Cambium 2024 Data. Retrieved September 2, 2025, from data.nrel.gov: https://data.nrel.gov/submissions/289

Guidehouse. (2021). 2021 Energy Waste Reduction and Demand Response Statewide Potential Study. State of Michigan. Retrieved September 10, 2025, from https://www.michigan.gov/mpsc/commission/workgroups/2021-energy-waste-reductionand-demand-response-statewide-potential-study

Industrial Training and Assessment Centers. (2025). Retrieved September 10, 2025, from Energy.gov: https://www.energy.gov/mesc/industrial-assessment-centers-iacs

Michigan Legislature. (n.d.). Michigan Legislature MCL - Index of Chapter 460, MCL - Index of Chapter 460. Retrieved September 9, 2025, from Legislature.mi.gov: https://www.legislature.mi.gov/Laws/Index?ObjectName=mcl-chap460

National Renewable Energy Lab. (2024). ResStock- NREL. Retrieved September 15, 2025, from nrel.gov: resstock.nrel.gov

Parker, A. H. (2024). ComStock reference documentation: 2024 Release 2. Retrieved from https://nrel.github.io/ComStock.github.io/assets/files/comstock_reference_documentation_2024_2.pdf

RSMeans. (n.d.). RSMeans Data Online. Retrieved October 2025, from RSMeans Data Online: https://www.rsmeans.com/

State of Michigan. (2024). Michigan Comprehensive Laws Annotated Chapter 460. West.

State of Michigan. (2025). Michigan Energy Measures Database. (M. P. (MPSC), Producer) Retrieved from michigan.gov/mpsc: https://www.michigan.gov/mpsc/regulatory/energyoptimization/michigan-energy-measures-database

State of Michigan. (2025). Michigan GIS Open Data. Retrieved September 2025, from https://gis-michigan.opendata.arcgis.com/

U.S. Census Bureau. (2024). 2022 Economic Census. (US Department of Commerce) Retrieved September 2025, from https://www.census.gov/programs-surveys/economic-census.htm

U.S. Department of Energy. (2024). 2020 U.S. Lighting Market Characterization DOE/GO-102022-5695. Solid State Lighting Program. Retrieved September 2025, from energy.gov: https://www.energy.gov/sites/default/files/2024-04/2020%20U.S.%20Lighting%20Market%20Characterization.pdf

U.S. Department of Treasury. (2025, February 8). Daily Treasury Par Real Yield Curve Rates. Retrieved September 10, 2025, from Home.treasury.gov: https://home.treasury.gov/resource-center/data-chart-center/interestrates/TextView?type=daily_treasury_real_yield_curve&field_tdr_date_value=2024

U.S. Energy Information Administration. (2022). 2018 Commercial Buildings Energy Consumption Survey (CBECS). Retrieved September 7, 2025, from eia.gov: https://www.eia.gov/consumption/commercial/

U.S. Energy Information Administration. (2025). 2022 Manufacturing Energy Consumption Survey (MECS). Retrieved September 7, 2025, from eia.gov: https://www.eia.gov/consumption/manufacturing/

U.S. Energy Information Administration. (2025, October). Annual Electric Power Industry Report, form EIA-861 detailed data files - U.S. Energy Information Administration (EIA). Annual Electric Power Industry Report, Form EIA-861 Detailed Data Files. Retrieved October 7, 2025, from eia.gov: https://www.eia.gov/electricity/data/eia861/

U.S. Energy Information Administration. (2025, April 16). Annual Energy Outlook 2025. Retrieved from eia.gov: https://www.eia.gov/outlooks/aeo/

U.S. Energy Information Administration. (2025). Natural Gas. U.S. Natural Gas Prices. Retrieved October 2025, from eia.gov: https://www.eia.gov/dnav/ng/ng_pri_sum_dcu_nus_m.htm

U.S. Energy Information Administration. (n.d.). Natural Gas Michigan Natural Gas Consumption by End Use. Retrieved September 15, 2025, from eia.gov: https://www.eia.gov/dnav/ng/ng_cons_sum_dcu_SMI_a.htm

U.S. Environmental Protection Agency. (2023). External Peer Review of EPA Draft Technical Report. Report on the Social Cost of Greenhouse Gases. Retrieved September 8, 2025, from https://www.epa.gov/system/files/documents/2023-12/charge-questions-for-draft-epascghg-report_final.pdf

U.S. Environmental Protection Agency. (2025). GHG Emissions Factors Hub. Retrieved September 5, 2025, from epa.gov: https://www.epa.gov/climateleadership/ghg-emission-factors-hub

Williams, S. G. (2021). Michigan Upper Peninsula Housing Baseline Study [Final report]. State of Michigan, Michigan Public Service Commission. Retrieved September 2025, from https://www.michigan.gov/-/media/Project/Websites/mpsc/workgroups/EWR_Collaborative/2021/Michigan_UP_Baseline_Study.pdf?rev=22561269e6084834919925c6cd4e3d27
"""


def normalize_url(url):
    """Normalize URL for comparison by removing www, trailing slashes, etc."""
    url = url.strip().rstrip('/')
    parsed = urlparse(url)
    # Remove www. and lowercase the domain
    netloc = parsed.netloc.lower().replace('www.', '')
    # Reconstruct URL without fragment
    normalized = urlunparse((parsed.scheme, netloc, parsed.path, '', '', ''))
    return normalized.rstrip('/')

# Create lookup dictionary from existing references
reference_lookup = {}

for ref_block in existing_references.strip().split('\n\n'):
    # Find all URLs in the reference block
    urls = re.findall(r'https?://[^\s\)]+', ref_block)
    
    # Clean the citation (remove extra URLs that appear at the end)
    citation = ref_block.strip()
    
    # For each URL found, add to lookup
    for url in urls:
        normalized = normalize_url(url)
        if normalized:
            reference_lookup[normalized] = citation

print(f"Loaded {len(reference_lookup)} pre-formatted references")
print(f"Sample URLs: {list(reference_lookup.keys())[:3]}")

Loaded 24 pre-formatted references
Sample URLs: ['https://censusreporter.org/:', 'https://censusreporter.org/tables/B25001', 'https://michigan.gov/mpsc/-/media/Project/Websites/mpsc/regulatory/reports/pa295-ewr/2024-EWR-Report-to-theLegislature.pdf']


In [76]:


def url_to_apa_format(url):
    """
    Convert a URL to APA citation format by extracting metadata from the webpage.
    First checks if URL exists in pre-formatted reference lookup.
    
    APA Format: Author. (Year, Month Day). Title. Site Name. URL
    """
    # First, check if we have a pre-formatted reference for this URL
    normalized = normalize_url(url)
    if normalized in reference_lookup:
        print(f"✓ Found pre-formatted reference for: {url[:60]}...")
        return reference_lookup[normalized]
    
    # If not found, scrape the webpage
    print(f"⚠ Scraping new reference for: {url[:60]}...")
    
    try:
        # Fetch the webpage
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Extract title - try multiple sources
        title_text = None
        
        # Try Open Graph title
        og_title = soup.find('meta', attrs={'property': 'og:title'})
        if og_title and og_title.get('content'):
            title_text = og_title.get('content').strip()
        
        # Try Twitter title
        if not title_text:
            twitter_title = soup.find('meta', attrs={'name': 'twitter:title'})
            if twitter_title and twitter_title.get('content'):
                title_text = twitter_title.get('content').strip()
        
        # Try regular title tag
        if not title_text:
            title = soup.find('title')
            if title:
                title_text = title.string.strip()
        
        # Try h1 tag
        if not title_text:
            h1 = soup.find('h1')
            if h1:
                title_text = h1.get_text().strip()
        
        if not title_text:
            title_text = 'No title'
        
        # Extract author - try multiple sources
        author = None
        
        # Try JSON-LD structured data
        json_ld = soup.find('script', type='application/ld+json')
        if json_ld:
            try:
                data = json.loads(json_ld.string)
                if isinstance(data, list):
                    data = data[0]
                if 'author' in data:
                    if isinstance(data['author'], dict):
                        author = data['author'].get('name')
                    else:
                        author = data['author']
            except:
                pass
        
        # Try meta tags
        if not author:
            author_meta = soup.find('meta', attrs={'name': 'author'}) or \
                         soup.find('meta', attrs={'property': 'article:author'}) or \
                         soup.find('meta', attrs={'name': 'DC.creator'}) or \
                         soup.find('meta', attrs={'property': 'og:article:author'})
            if author_meta and author_meta.get('content'):
                author = author_meta.get('content')
        
        # Try byline class
        if not author:
            byline = soup.find(class_=re.compile('byline|author', re.I))
            if byline:
                author = byline.get_text().strip()
                # Clean common prefixes
                author = re.sub(r'^(by|author):?\s*', '', author, flags=re.I).strip()
        
        # Extract publish date - try multiple sources
        date = None
        access_date = datetime.now().strftime('%Y, %B %d')
        
        # Try JSON-LD
        if json_ld:
            try:
                data = json.loads(json_ld.string)
                if isinstance(data, list):
                    data = data[0]
                date_str = data.get('datePublished') or data.get('dateCreated')
                if date_str:
                    date_obj = datetime.fromisoformat(date_str.replace('Z', '+00:00'))
                    date = date_obj.strftime('%Y, %B %d')
            except:
                pass
        
        # Try meta tags
        if not date:
            date_meta = soup.find('meta', attrs={'property': 'article:published_time'}) or \
                       soup.find('meta', attrs={'name': 'date'}) or \
                       soup.find('meta', attrs={'name': 'publishdate'}) or \
                       soup.find('meta', attrs={'property': 'og:published_time'}) or \
                       soup.find('meta', attrs={'name': 'DC.date'}) or \
                       soup.find('meta', attrs={'itemprop': 'datePublished'})
            
            if date_meta and date_meta.get('content'):
                try:
                    date_str = date_meta.get('content')
                    date_obj = datetime.fromisoformat(date_str.replace('Z', '+00:00'))
                    date = date_obj.strftime('%Y, %B %d')
                except:
                    pass
        
        # Try time tags
        if not date:
            time_tag = soup.find('time', attrs={'datetime': True})
            if time_tag:
                try:
                    date_str = time_tag['datetime']
                    date_obj = datetime.fromisoformat(date_str.replace('Z', '+00:00'))
                    date = date_obj.strftime('%Y, %B %d')
                except:
                    pass
        
        # Extract site name from domain or meta tags
        domain = urlparse(url).netloc
        
        # Try Open Graph site name
        og_site = soup.find('meta', attrs={'property': 'og:site_name'})
        if og_site and og_site.get('content'):
            site_name = og_site.get('content')
        else:
            site_name = domain.replace('www.', '').split('.')[0].capitalize()
        
        # Build APA citation
        citation_parts = []
        
        if author:
            citation_parts.append(f"{author}.")
        
        if date:
            citation_parts.append(f"({date}).")
        else:
            citation_parts.append(f"(n.d.).")  # no date
        
        citation_parts.append(f"*{title_text}*.")
        citation_parts.append(f"{site_name}.")
        
        # Add retrieval date if no publish date
        if not date:
            citation_parts.append(f"Retrieved {access_date}, from")
        
        citation_parts.append(url)
        
        return ' '.join(citation_parts)
    
    except Exception as e:
        # If scraping fails, return a basic format with just the URL
        print(f"✗ Error scraping {url[:60]}...: {str(e)}")
        domain = urlparse(url).netloc
        site_name = domain.replace('www.', '')
        access_date = datetime.now().strftime('%B %d, %Y')
        return f"(n.d.). *Retrieved from {site_name}*. Retrieved {access_date}, from {url}"

# Apply the function to create APA formatted citations
print("Processing references...")
references_df['apa_citation'] = references_df['reference'].apply(url_to_apa_format)

# Display results
print(f"\n{'='*80}")
print(f"Processed {len(references_df)} references")
print(f"{'='*80}\n")
print(references_df[['reference', 'apa_citation']].head(10))

Processing references...
⚠ Scraping new reference for: https://www.ilsag.info/technical-reference-manual/...
⚠ Scraping new reference for: https://www.masssavedata.com/...
⚠ Scraping new reference for: https://censusreporter.org/...
⚠ Scraping new reference for: https://www.michigan.gov/mpsc/-/media/Project/Websites/mpsc/...


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.



Processed 4 references

                                           reference  \
0  https://www.ilsag.info/technical-reference-man...   
1                      https://www.masssavedata.com/   
2                        https://censusreporter.org/   
3  https://www.michigan.gov/mpsc/-/media/Project/...   

                                        apa_citation  
0  (n.d.). *Illinois Statewide Technical Referenc...  
1  (n.d.). *Mass Save Data*. Masssavedata. Retrie...  
2  (n.d.). *Census Reporter: Making Census Data E...  
3  (n.d.). *No title*. Michigan. Retrieved 2026, ...  


In [77]:
# Create a new Word document
doc = Document()

# Add title
title = doc.add_heading('References', level=1)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Add each citation as a paragraph with hanging indent (APA style)
for idx, row in references_df.iterrows():
    # Add the APA citation
    p = doc.add_paragraph()
    p.paragraph_format.left_indent = Inches(0.5)
    p.paragraph_format.first_line_indent = Inches(-0.5)
    p.paragraph_format.space_after = Pt(0)
    
    # Parse and format the citation (remove markdown italics for Word)
    citation = row['apa_citation'].replace('*', '')
    p.add_run(citation)

# Save the document
output_path = '050_output/references_apa.docx'
doc.save(output_path)
print(f"References saved to {output_path}")

PermissionError: [Errno 13] Permission denied: '050_output/references_apa.docx'

In [ ]:
# Debug: Check which URLs from the dataframe match the pre-formatted references
print("Checking URL matching...\n")

matched_count = 0
unmatched_count = 0

for idx, row in references_df.iterrows():
    url = row['reference']
    normalized = normalize_url(url)
    
    if normalized in reference_lookup:
        matched_count += 1
        print(f"✓ MATCHED: {url[:80]}")
    else:
        unmatched_count += 1
        print(f"✗ NOT MATCHED: {url[:80]}")
        # Show what normalized version looks like
        print(f"  Normalized to: {normalized[:80]}")

print(f"\n{'='*80}")
print(f"Summary: {matched_count} matched, {unmatched_count} not matched")
print(f"{'='*80}")

# Also show a few examples from the lookup dictionary
print(f"\nSample pre-formatted reference URLs in lookup:")
for i, url in enumerate(list(reference_lookup.keys())[:5]):
    print(f"  {i+1}. {url}")

Checking URL matching...

✗ NOT MATCHED: https://www.ilsag.info/technical-reference-manual/
  Normalized to: https://ilsag.info/technical-reference-manual
✗ NOT MATCHED: https://www.masssavedata.com/
  Normalized to: https://masssavedata.com
✗ NOT MATCHED: https://censusreporter.org/
  Normalized to: https://censusreporter.org
✗ NOT MATCHED: https://www.michigan.gov/mpsc/-/media/Project/Websites/mpsc/regulatory/reports/p
  Normalized to: https://michigan.gov/mpsc/-/media/Project/Websites/mpsc/regulatory/reports/pa295

Summary: 0 matched, 4 not matched

Sample pre-formatted reference URLs in lookup:
  1. https://censusreporter.org/:
  2. https://censusreporter.org/tables/B25001
  3. https://michigan.gov/mpsc/-/media/Project/Websites/mpsc/regulatory/reports/pa295-ewr/2024-EWR-Report-to-theLegislature.pdf
  4. https://michigan.gov/mpsc/-/media/Project/Websites/mpsc/regulatory/ewr/MEMDand-BRM/Michigan-Baseline-Housing-StudyFinal-2021-05-25.pdf
  5. https://data.nrel.gov/submissions/289
